# Clasificación de Géneros de Películas

### Parte 4: Resultados y Conclusiones

---

Análisis comparativo de los 9 modelos, evaluación del campeón por género y ejemplo de inferencia sobre sinopsis nuevas.

In [ ]:
# Librerías
import warnings
warnings.filterwarnings('ignore')
import re, ast
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
sns.set_theme(style='whitegrid')

In [ ]:
# Cargar datos y reproducir pipeline completo
dataTraining = pd.read_csv('data/dataTraining.csv', encoding='UTF-8', index_col=0)
dataTesting  = pd.read_csv('data/dataTesting.csv',  encoding='UTF-8', index_col=0)

dataTraining['genres'] = dataTraining['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
dataTraining['genres'] = dataTraining['genres'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
dataTraining = dataTraining.drop_duplicates()
dataTraining['genres'] = dataTraining['genres'].apply(lambda x: list(x) if isinstance(x, tuple) else x)

stop_words = set(stopwords.words('english'))

def remove_tags(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[\W_]', ' ', text)
    return text.lower()

def remove_stopwords(text):
    return ' '.join(w for w in text.split() if w not in stop_words)

w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer  = nltk.stem.WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join(lemmatizer.lemmatize(w) for w in w_tokenizer.tokenize(text))

for df in [dataTraining, dataTesting]:
    df['plot'] = df['plot'].apply(remove_tags).apply(remove_stopwords).apply(lemmatize_text)

mlb  = MultiLabelBinarizer()
y    = mlb.fit_transform(dataTraining['genres'])
cols = [f'p_{g}' for g in mlb.classes_]

print(f'Datos listos: {len(dataTraining)} películas | {len(mlb.classes_)} géneros')

In [ ]:
# Entrenar modelo campeón sobre el set completo (split 67/33)
tfidf = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,3))
X     = tfidf.fit_transform(dataTraining['plot'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=100, C=1, penalty='l2', solver='liblinear',
                       random_state=42, n_jobs=-1)
)
clf.fit(X_train, y_train)

y_pred = clf.predict_proba(X_test)
print(f'ROC-AUC macro: {roc_auc_score(y_test, y_pred, average="macro"):.4f}')

### 1. Comparación de modelos

In [ ]:
# Tabla resumen con resultados de los 9 modelos
resultados = pd.DataFrame({
    'Modelo': [
        'LR trigramas + lematización', 'Feature Union', 'Ridge Classifier',
        'LR baseline', 'Red Neuronal Densa', 'Naive Bayes (ComplementNB)',
        'BiLSTM + GloVe', 'LightGBM', 'Random Forest'
    ],
    'Representación': [
        'TF-IDF', 'TF-IDF + longitud', 'TF-IDF',
        'TF-IDF', 'TF-IDF', 'CountVectorizer',
        'GloVe 100d', 'TF-IDF', 'TF-IDF'
    ],
    'ROC-AUC macro': [
        0.888, 0.886, 0.884, 0.884, 0.880, 0.876, 0.874, 0.830, 0.820
    ]
})

resultados.style.background_gradient(subset=['ROC-AUC macro'], cmap='Greens')

In [ ]:
# Gráfico comparativo
plt.figure(figsize=(11, 5))
colors = ['#2ecc71' if m == 'LR trigramas + lematización' else '#3498db'
          for m in resultados['Modelo']]
bars = plt.barh(resultados['Modelo'], resultados['ROC-AUC macro'], color=colors)
plt.xlabel('ROC-AUC macro')
plt.title('Comparación de modelos')
plt.xlim(0.79, 0.90)
plt.gca().invert_yaxis()
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.001, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

### 2. ROC por género — modelo campeón

In [ ]:
# Calcular AUC individual para cada uno de los 24 géneros
fpr_d, tpr_d, auc_d = {}, {}, {}
n = y_pred.shape[1]

for i in range(n):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_test[:, i], y_pred[:, i])
    auc_d[i] = auc(fpr_d[i], tpr_d[i])

# Macro-average: interpolar y promediar todas las curvas
all_fpr   = np.unique(np.concatenate([fpr_d[i] for i in range(n)]))
mean_tpr  = sum(np.interp(all_fpr, fpr_d[i], tpr_d[i]) for i in range(n)) / n
auc_macro = auc(all_fpr, mean_tpr)

# Gráfico con curvas por clase + macro-average
plt.figure(figsize=(12, 9))
plt.plot(all_fpr, mean_tpr, 'navy', linestyle=':', lw=3,
         label=f'Macro-average (AUC={auc_macro:.3f})')
for i in range(n):
    plt.plot(fpr_d[i], tpr_d[i], lw=1,
             label=f'{mlb.classes_[i]} ({auc_d[i]:.2f})')
plt.plot([0,1],[0,1],'--',color='gray',lw=1.5)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Curvas ROC — Modelo campeón')
plt.legend(loc='lower right', fontsize='x-small', ncol=2)
plt.tight_layout(); plt.show()

### 3. AUC por género

In [ ]:
# Ordenar géneros por AUC para identificar los más y menos predecibles
auc_serie = pd.Series(
    {mlb.classes_[i]: auc_d[i] for i in range(n)}
).sort_values(ascending=True)

colors = ['#e74c3c' if v < 0.85 else '#2ecc71' if v > 0.95 else '#3498db'
          for v in auc_serie]

plt.figure(figsize=(10, 7))
bars = plt.barh(auc_serie.index, auc_serie.values, color=colors)
plt.axvline(auc_macro, color='navy', linestyle='--', lw=1.5,
            label=f'Macro-avg: {auc_macro:.3f}')
plt.xlabel('ROC-AUC')
plt.title('AUC por género')
plt.xlim(0.5, 1.01)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.003, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=8)
plt.legend(); plt.tight_layout(); plt.show()

### 4. Inferencia — nuevas sinopsis

In [ ]:
def predecir_generos(sinopsis, top_n=5):
    # Aplicar el mismo pipeline de preprocesamiento al texto nuevo
    texto = lemmatize_text(remove_stopwords(remove_tags(sinopsis)))
    X_new = tfidf.transform([texto])
    probs = clf.predict_proba(X_new)[0]
    resultado = pd.DataFrame({'genero': mlb.classes_, 'probabilidad': probs})
    return resultado.sort_values('probabilidad', ascending=False).head(top_n).reset_index(drop=True)

# Ejemplo 1: thriller de ciencia ficción
sinopsis_1 = """
A rogue scientist discovers the government has been secretly experimenting with alien DNA,
turning soldiers into unstoppable killers. Racing against time, she must expose the
conspiracy before the next test subject is unleashed.
"""
print('Thriller / Sci-Fi:')
print(predecir_generos(sinopsis_1).to_string(index=False))

In [ ]:
# Ejemplo 2: drama romántico
sinopsis_2 = """
Two strangers meet on a cross-country train and fall in love over three days. When
the journey ends, they must decide whether to leave their old lives behind.
"""
print('Drama / Romance:')
print(predecir_generos(sinopsis_2).to_string(index=False))

In [ ]:
# Generar predicciones sobre el set de test completo y guardar
X_final    = tfidf.transform(dataTesting['plot'])
y_pred_fin = clf.predict_proba(X_final)

submission = pd.DataFrame(y_pred_fin, index=dataTesting.index, columns=cols)
submission.to_csv('data/predicciones_finales.csv', index_label='ID')

print(f'Predicciones guardadas: {submission.shape}')
submission.head(3)

### 5. Conclusiones

- **TF-IDF + LR con trigramas** es el método óptimo. La frontera de decisión del problema es lineal en el espacio de términos TF-IDF.
- **Trigramas capturan frases clave** por género: `"serial killer"`, `"outer space"`, `"romantic comedy"`.
- **LightGBM y Random Forest underperforman** en matrices dispersas: los árboles necesitan muchos splits para aproximar lo que LR resuelve directamente.
- **GloVe no supera TF-IDF**: las sinopsis son cortas y el vocabulario temático se beneficia del matching exacto más que de la generalización semántica.
- **Géneros poco frecuentes** (News, Film-Noir, Short) tienen AUC más bajo: el desbalanceo de clases es el principal reto del problema.